# DS110 — Introduction to Data Science
## Phase 3: Statistical Modelling & Machine Learning
### *Feeding the Few: Mapping Inequality Behind Pakistan's Hunger Crisis*

**Group 06** | Aamna Nosheen (2503600) · Kinza Eman (2503644) · Zunaira Hassan (2503650)
Air University, Islamabad — May 2026

---

### Phase 3 Overview
This notebook applies three ML/statistical algorithms to the cleaned datasets from Phase 2, directly addressing the research questions set out in Phase 1.

| # | Algorithm | Type | Research Question Addressed |
|---|-----------|------|-----------------------------|
| 1 | **Linear Regression** | In-class (required) | Can city-level price features predict food basket cost? |
| 2 | **Gaussian Naive Bayes** | In-class (required) | Can we classify a city's food stress level from price data? |
| 3 | **K-Means Clustering** | Outside class | Can cities be grouped into distinct food insecurity profiles? |

---

## Section 0 — Library Imports

In [ ]:
# ── Core libraries ───────────────────────────────────────
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import seaborn as sns
from scipy import stats

# ── Machine Learning ──────────────────────────────────────
from sklearn.linear_model import LinearRegression
from sklearn.naive_bayes import GaussianNB
from sklearn.cluster import KMeans
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import (
    r2_score, mean_absolute_error, mean_squared_error,
    accuracy_score, classification_report,
    confusion_matrix, silhouette_score
)
from sklearn.model_selection import LeaveOneOut, cross_val_score

# ── Plot style ────────────────────────────────────────────
plt.rcParams.update({
    'font.family'       : 'DejaVu Sans',
    'axes.spines.top'   : False,
    'axes.spines.right' : False,
    'axes.grid'         : True,
    'grid.alpha'        : 0.3,
    'figure.dpi'        : 130,
})

# Colour palette (consistent across all Phase 3 charts)
TEAL   = '#1A5276'
RED    = '#E74C3C'
ORANGE = '#E67E22'
GREEN  = '#1E8449'
PURPLE = '#8E44AD'
GREY   = '#5D6D7E'

print("All libraries imported successfully")

## Section 1 — Load Fixed Datasets

In [ ]:
# ── Change this path to match your Kaggle dataset path ───
DATA = '/kaggle/input/YOUR-FIXED-DATASET-PATH/'
# Example: '/kaggle/input/kinzaemannn-fixed-datasets/'

spi     = pd.read_csv(DATA + 'spi_prices_clean_FIXED.csv',  parse_dates=['Date'])
fao     = pd.read_csv(DATA + 'fao_food_supply_clean_FIXED.csv')
wb      = pd.read_csv(DATA + 'worldbank_clean_FIXED.csv')
wb_real = pd.read_csv(DATA + 'worldbank_real_surveys_only.csv')

# ── Filter to reliable food rows only ─────────────────────
# (removes Quetta Wheat Flour unreliable rows AND non-food items)
NON_FOOD = ['Electricity','Gas','Petrol','Diesel','Lpg','Telephone',
            'Cigarettes','Georgette','Lawn','Long Cloth','Shirting',
            'Sandal','Chappal','Energy Saver','Firewood',
            'Match Box','Washing Soap','Toilet Soap']
pattern      = '|'.join(NON_FOOD)
spi_reliable = spi[spi['Data_Reliable'] == True].copy()
spi_food     = spi_reliable[
    ~spi_reliable['Item'].str.contains(pattern, case=False, na=False)
].copy()

print(f"SPI food rows  : {len(spi_food):,}")
print(f"Unique cities  : {spi_food['City'].nunique()}")
print(f"Unique items   : {spi_food['Item'].nunique()}")
print(f"Date range     : {spi_food['Date'].min().date()} → {spi_food['Date'].max().date()}")
print(f"FAO shape      : {fao.shape}")
print(f"WB real surveys: {wb_real.shape}")

## Section 2 — Feature Engineering

Before running any model, we build a city-level feature matrix.
Each row represents one city. Features capture:
- **Average price level** — how expensive food is in general
- **Price volatility** — how unstable prices are (standard deviation)
- **Provincial wage** — purchasing power proxy
- **Basket cost** — total cost of 10 essential food items
- **Affordability %** — basket cost as a share of monthly wage
- **Item-level CV** — coefficient of variation for 3 volatile items

In [ ]:
# ── Define 10-item food basket (consistent with Phase 2 EDA 3) ──
BASKET_ITEMS = [
    'Wheat Flour Bag',
    'Rice Irri-6/9 (Sindh/Punjab)',
    'Milk Fresh (Un-Boiled)',
    'Eggs Hen (Farm)',
    'Pulse Gram',
    'Sugar Refined',
    'Cooking Oil Dalda Or Other Similar',
    'Tomatoes',
    'Onions',
    'Chicken Farm Broiler (Live)',
]

# Provincial wage benchmarks (PBS Labour Force Survey 2022-23)
PROV_WAGES = {
    'ICT'         : 53000,
    'Punjab'      : 43000,
    'Sindh'       : 47000,
    'KPK'         : 32000,
    'Balochistan' : 28000,
}

# ── Build city-level summary ───────────────────────────────
city_df = spi_food.groupby('City').agg(
    Avg_Price        = ('Avg_Price', 'mean'),
    Price_Volatility = ('Avg_Price', 'std'),
    Province         = ('Province', 'first'),
).reset_index()

city_df['Wage'] = city_df['Province'].map(PROV_WAGES)

# ── Basket cost per city (avg unit price × 10 items) ──────
basket_city = (
    spi_food[spi_food['Item'].isin(BASKET_ITEMS)]
    .groupby('City')['Avg_Price']
    .mean() * 10          # sum of average unit prices across basket
)
city_df = city_df.merge(basket_city.rename('Basket_Cost'), on='City')

# ── Affordability: basket cost as % of monthly wage ───────
city_df['Affordability_Pct'] = (
    city_df['Basket_Cost'] / city_df['Wage'] * 100
).round(2)

# ── Item-level CV per city (for Naive Bayes features) ─────
for item_name, col_name in [
    ('Wheat Flour Bag',        'Wheat_CV'),
    ('Tomatoes',               'Tomato_CV'),
    ('Milk Fresh (Un-Boiled)', 'Milk_CV'),
]:
    sub = (
        spi_food[spi_food['Item'] == item_name]
        .groupby('City')['Avg_Price']
        .apply(lambda x: x.std() / x.mean() * 100)
    )
    city_df = city_df.merge(sub.rename(col_name), on='City', how='left')

print("City-level feature matrix:")
print(city_df[['City', 'Province', 'Avg_Price', 'Basket_Cost',
               'Affordability_Pct', 'Wage']].to_string())
print(f"\nShape: {city_df.shape}")

---
## Section 3 — Model 1: Linear Regression (In-Class Algorithm)

### Why Linear Regression?
Linear Regression models the relationship between one or more **predictor variables** (X)
and a **continuous target variable** (y) by fitting a straight line (or hyperplane) that
minimises the sum of squared residuals.

**Mathematical form:**
> `Basket_Cost = β₀ + β₁(Avg_Price) + β₂(Price_Volatility) + β₃(Wage) + ε`

**Why it fits our problem:**
- Our target (monthly food basket cost in PKR) is a continuous numeric variable
- We have three quantitative predictors that logically should influence basket cost:
  the general price level in a city, how volatile prices are, and local wage levels
- Linear Regression is interpretable — each coefficient tells us exactly how much
  basket cost changes for a one-unit increase in that predictor

**Assumptions:**
1. Linear relationship between predictors and target
2. Residuals are approximately normally distributed
3. No perfect multicollinearity between predictors

**Justification for this choice over time-series regression:**
The SPI dataset covers Aug 2023 – Aug 2025, a period that includes a major structural break
(wheat price government intervention, mid-2024). A city-cross-sectional regression is more
stable and interpretable than a time series model with a known structural break.

In [ ]:
# ════════════════════════════════════════════════════════════
#  MODEL 1: LINEAR REGRESSION
#  Target  : Basket_Cost (PKR) — total 10-item basket per city
#  Features: Avg_Price, Price_Volatility, Wage
# ════════════════════════════════════════════════════════════

# ── Feature matrix and target ─────────────────────────────
FEATURES_LR = ['Avg_Price', 'Price_Volatility', 'Wage']
X_lr = city_df[FEATURES_LR].values
y_lr = city_df['Basket_Cost'].values

# ── Fit model ─────────────────────────────────────────────
lr = LinearRegression()
lr.fit(X_lr, y_lr)

# ── Predictions and residuals ─────────────────────────────
y_pred_lr        = lr.predict(X_lr)
residuals_lr     = y_lr - y_pred_lr

# ── Evaluation metrics ────────────────────────────────────
r2   = r2_score(y_lr, y_pred_lr)
mae  = mean_absolute_error(y_lr, y_pred_lr)
rmse = np.sqrt(mean_squared_error(y_lr, y_pred_lr))

# ── Store predictions in city_df for later plotting ───────
city_df['LR_Predicted'] = y_pred_lr.round(1)
city_df['LR_Residual']  = residuals_lr.round(1)

print("=" * 55)
print("MODEL 1: LINEAR REGRESSION — Results")
print("=" * 55)
print(f"  R² (coefficient of determination) : {r2:.4f}")
print(f"  MAE (mean absolute error)          : PKR {mae:.1f}")
print(f"  RMSE (root mean squared error)     : PKR {rmse:.1f}")
print()
print("Regression Equation:")
print(f"  Basket_Cost = {lr.intercept_:.1f}")
for feat, coef in zip(FEATURES_LR, lr.coef_):
    sign = '+' if coef >= 0 else '-'
    print(f"              {sign} {abs(coef):.4f} × {feat}")
print()
print("Interpretation of Coefficients:")
print(f"  Avg_Price coef = {lr.coef_[0]:.3f}")
print("  → Each PKR 1 increase in a city's average item price")
print(f"    increases the basket cost by PKR {lr.coef_[0]:.2f}")
print()
print(f"  Price_Volatility coef = {lr.coef_[1]:.3f}")
print("  → Higher price volatility is associated with higher basket cost.")
print("    More unstable markets tend to have higher average prices.")
print()
print(f"  Wage coef = {lr.coef_[2]:.5f}")
print("  → Wage has minimal direct effect on basket cost")
print("    (basket cost is a market phenomenon, wages don't set prices)")
print()
print("Actual vs Predicted:")
display_df = city_df[['City','Province','Basket_Cost','LR_Predicted','LR_Residual']].copy()
display_df.columns = ['City','Province','Actual_PKR','Predicted_PKR','Residual_PKR']
print(display_df.to_string(index=False))

In [ ]:
# ── VISUALISATION 1a: Actual vs Predicted + Residuals ─────

fig, axes = plt.subplots(1, 2, figsize=(14, 6))

# Panel 1: Actual vs Predicted scatter
ax = axes[0]
ax.scatter(y_lr, y_pred_lr, color=TEAL, s=90, edgecolors='white',
           linewidth=1.5, zorder=5, label='Cities')

# Perfect prediction line (45°)
min_v = min(y_lr.min(), y_pred_lr.min()) - 200
max_v = max(y_lr.max(), y_pred_lr.max()) + 200
ax.plot([min_v, max_v], [min_v, max_v], color=RED,
        linewidth=2, linestyle='--', label='Perfect prediction (y=x)')

# Annotate outlier cities
for _, row in city_df.iterrows():
    if abs(row['LR_Residual']) > 250:
        ax.annotate(row['City'],
                    xy=(row['Basket_Cost'], row['LR_Predicted']),
                    xytext=(8, -12), textcoords='offset points',
                    fontsize=8, color=GREY)

ax.set_xlabel('Actual Basket Cost (PKR)', fontsize=12)
ax.set_ylabel('Predicted Basket Cost (PKR)', fontsize=12)
ax.set_title('Model 1: Linear Regression\nActual vs Predicted Basket Cost',
             fontsize=12, fontweight='bold')
ax.legend(fontsize=9)

# R² annotation box
ax.text(0.05, 0.90, f'R² = {r2:.4f}\nMAE = PKR {mae:.0f}\nRMSE = PKR {rmse:.0f}',
        transform=ax.transAxes, fontsize=10,
        bbox=dict(boxstyle='round', facecolor='#D6EAF8', alpha=0.9))

# Panel 2: Residual bar chart
ax2 = axes[1]
residual_order = city_df.sort_values('LR_Residual')
bar_colors = [RED if r < 0 else GREEN for r in residual_order['LR_Residual']]
bars = ax2.barh(residual_order['City'], residual_order['LR_Residual'],
                color=bar_colors, edgecolor='white')

# Value labels
for bar, val in zip(bars, residual_order['LR_Residual']):
    offset = 5 if val >= 0 else -60
    ax2.text(val + offset, bar.get_y() + bar.get_height()/2,
             f'{val:+.0f}', va='center', fontsize=8)

ax2.axvline(0, color='black', linewidth=1.2)
ax2.set_xlabel('Residual (Actual − Predicted) PKR', fontsize=11)
ax2.set_title('Residual Analysis by City\n(Green = over-predicted; Red = under-predicted)',
              fontsize=12, fontweight='bold')

green_patch = mpatches.Patch(color=GREEN, label='Model under-predicted (actual > predicted)')
red_patch   = mpatches.Patch(color=RED,   label='Model over-predicted (actual < predicted)')
ax2.legend(handles=[green_patch, red_patch], fontsize=8)

plt.tight_layout()
plt.suptitle('Linear Regression — Basket Cost Prediction Results',
             fontsize=14, fontweight='bold', y=1.02)
plt.savefig('lr_actual_vs_predicted.png', dpi=150, bbox_inches='tight')
plt.show()

print("\nInsight: R² = 0.957 means the model explains 95.7% of variation in")
print("basket cost across cities using just 3 features.")
print("Average price level is the strongest predictor (coefficient = 2.47).")
print("Cities like Quetta and Bannu are pulled from the main cluster by low")
print("wages and prices — the model handles them well (low residuals).")

In [ ]:
# ── VISUALISATION 1b: Feature Importance (Standardised Coefficients) ─

# Standardise features to compare coefficients on equal footing
from sklearn.preprocessing import StandardScaler
scaler_lr = StandardScaler()
X_scaled_lr = scaler_lr.fit_transform(X_lr)
lr_scaled   = LinearRegression()
lr_scaled.fit(X_scaled_lr, y_lr)

fig, ax = plt.subplots(figsize=(9, 4))

feature_labels = ['Average\nItem Price', 'Price\nVolatility', 'Provincial\nWage']
std_coefs      = lr_scaled.coef_
bar_colors     = [GREEN if c > 0 else RED for c in std_coefs]

bars = ax.bar(feature_labels, std_coefs, color=bar_colors,
              edgecolor='white', width=0.5)

for bar, val in zip(bars, std_coefs):
    ax.text(bar.get_x() + bar.get_width()/2,
            val + (50 if val > 0 else -100),
            f'{val:.1f}', ha='center', fontsize=11, fontweight='bold')

ax.axhline(0, color='black', linewidth=1)
ax.set_ylabel('Standardised Coefficient\n(effect per 1 std-dev change)', fontsize=11)
ax.set_title('Linear Regression — Standardised Feature Importance\n'
             '(Larger bar = stronger predictor of basket cost)',
             fontsize=12, fontweight='bold')

green_p = mpatches.Patch(color=GREEN, label='Positive effect (higher → higher cost)')
red_p   = mpatches.Patch(color=RED,   label='Negative effect (higher → lower cost)')
ax.legend(handles=[green_p, red_p], fontsize=9)

plt.tight_layout()
plt.savefig('lr_feature_importance.png', dpi=150, bbox_inches='tight')
plt.show()

print("Standardised coefficients tell us which feature matters most:")
for feat, coef in zip(feature_labels, std_coefs):
    print(f"  {feat.replace(chr(10),' ')} : {coef:.2f}")

---
## Section 4 — Model 2: Gaussian Naive Bayes (In-Class Algorithm)

### Why Naive Bayes?
Naive Bayes is a **probabilistic classifier** based on Bayes' Theorem with the
"naive" assumption that all features are conditionally independent given the class.

**Bayes' Theorem:**
> `P(Class | Features) ∝ P(Class) × P(Features | Class)`

**Gaussian Naive Bayes** assumes each feature follows a normal (Gaussian) distribution
within each class, making it ideal for continuous numeric features.

**Why it fits our problem:**
- We want to classify 16 Pakistani cities into **High Food Stress** or **Low Food Stress**
- The features (price level, volatility, affordability %) are continuous and approximately normally distributed
- With only 16 data points (cities), Naive Bayes performs well because it requires very few training samples
- The probabilistic output (P(High Stress)) is more informative than a hard binary label —
  it shows us *how confident* the model is about each city's risk level

**What is "Food Stress"?**
A city is classified as **High Stress** if its affordability ratio (basket cost as % of wage)
exceeds the median across all cities (15.49%). This threshold represents the point at which
food cost begins to meaningfully squeeze household budgets.

**Evaluation — Leave-One-Out Cross Validation (LOO-CV):**
With only 16 cities, standard train/test split would give too few test points to be reliable.
LOO-CV trains on 15 cities and tests on 1, repeated 16 times, giving an honest estimate
of generalisation accuracy.

In [ ]:
# ════════════════════════════════════════════════════════════
#  MODEL 2: GAUSSIAN NAIVE BAYES
#  Task   : Classify city as High Stress or Low Stress
#  Features: Avg_Price, Price_Volatility, Affordability_Pct,
#            Wheat_CV, Tomato_CV, Milk_CV
#  Target : Food_Stress (0 = Low, 1 = High)
# ════════════════════════════════════════════════════════════

# ── Define target: High Stress = affordability > median ───
MEDIAN_AFF = city_df['Affordability_Pct'].median()
city_df['Food_Stress']       = (city_df['Affordability_Pct'] > MEDIAN_AFF).astype(int)
city_df['Food_Stress_Label'] = city_df['Food_Stress'].map({0:'Low Stress', 1:'High Stress'})

print(f"Affordability median threshold : {MEDIAN_AFF:.2f}%")
print(f"Cities above threshold (High)  : {city_df['Food_Stress'].sum()}")
print(f"Cities below threshold (Low)   : {(city_df['Food_Stress']==0).sum()}")
print(f"Class balance                  : 50/50 (perfectly balanced)")

# ── Feature matrix ─────────────────────────────────────────
FEATURES_NB = ['Avg_Price', 'Price_Volatility', 'Affordability_Pct',
               'Wheat_CV', 'Tomato_CV', 'Milk_CV']
X_nb = city_df[FEATURES_NB].fillna(0).values
y_nb = city_df['Food_Stress'].values

# ── Fit Gaussian Naive Bayes ───────────────────────────────
nb = GaussianNB()
nb.fit(X_nb, y_nb)

# ── Leave-One-Out Cross Validation ────────────────────────
loo = LeaveOneOut()
loo_scores = cross_val_score(nb, X_nb, y_nb, cv=loo, scoring='accuracy')

# ── Training predictions and probabilities ────────────────
y_pred_nb  = nb.predict(X_nb)
proba_nb   = nb.predict_proba(X_nb)   # [[P(Low), P(High)], ...]

city_df['NB_Pred_Label'] = np.where(y_pred_nb == 1, 'High Stress', 'Low Stress')
city_df['NB_P_High']     = proba_nb[:, 1].round(3)

print("\n" + "=" * 55)
print("MODEL 2: GAUSSIAN NAIVE BAYES — Results")
print("=" * 55)
print(f"  LOO-CV Accuracy  : {loo_scores.mean():.3f} ± {loo_scores.std():.3f}")
print(f"  Training Accuracy: {accuracy_score(y_nb, y_pred_nb):.3f}")
print()
print("Classification Report:")
print(classification_report(y_nb, y_pred_nb,
                            target_names=['Low Stress', 'High Stress']))
print("Per-city predictions:")
disp_cols = ['City','Province','Affordability_Pct',
             'Food_Stress_Label','NB_Pred_Label','NB_P_High']
print(city_df[disp_cols].to_string(index=False))

In [ ]:
# ── VISUALISATION 2a: Confusion Matrix ────────────────────

fig, axes = plt.subplots(1, 2, figsize=(14, 6))

# Panel 1: Confusion Matrix
ax = axes[0]
cm = confusion_matrix(y_nb, y_pred_nb)
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
            xticklabels=['Low Stress', 'High Stress'],
            yticklabels=['Low Stress', 'High Stress'],
            linewidths=2, linecolor='white', ax=ax,
            annot_kws={'size': 18, 'weight': 'bold'})
ax.set_xlabel('Predicted Label', fontsize=12)
ax.set_ylabel('True Label', fontsize=12)
ax.set_title('Model 2: Naive Bayes\nConfusion Matrix',
             fontsize=12, fontweight='bold')

# Add metric annotations
ax.text(0.5, -0.18,
        f'Training Accuracy: {accuracy_score(y_nb, y_pred_nb):.1%}   '
        f'LOO-CV Accuracy: {loo_scores.mean():.1%}',
        transform=ax.transAxes, ha='center', fontsize=10,
        bbox=dict(boxstyle='round', facecolor='#D6EAF8', alpha=0.9))

# Panel 2: Probability bar chart — how confident is the model per city?
ax2 = axes[1]
city_sorted = city_df.sort_values('NB_P_High')
colors_prob = [RED if s == 1 else GREEN for s in city_sorted['Food_Stress']]
bars = ax2.barh(city_sorted['City'], city_sorted['NB_P_High'],
                color=colors_prob, edgecolor='white', alpha=0.85)

# Value labels
for bar, val in zip(bars, city_sorted['NB_P_High']):
    ax2.text(min(val + 0.02, 0.95), bar.get_y() + bar.get_height()/2,
             f'{val:.0%}', va='center', fontsize=9)

ax2.axvline(0.5, color='black', linewidth=1.5, linestyle='--',
            label='Decision boundary (50%)')
ax2.set_xlabel('P(High Food Stress)', fontsize=11)
ax2.set_title('Naive Bayes: Predicted Probability of High Food Stress\n'
              '(Actual High = Red, Actual Low = Green)',
              fontsize=11, fontweight='bold')
ax2.set_xlim(0, 1.05)
ax2.legend(fontsize=9)

plt.tight_layout()
plt.suptitle('Naive Bayes Classification — Food Stress Risk by City',
             fontsize=14, fontweight='bold', y=1.02)
plt.savefig('nb_results.png', dpi=150, bbox_inches='tight')
plt.show()

print("\nInterpretation:")
print("  Peshawar: P(High) = 100% — KPK wages (PKR 32,000) make 18.7% affordability catastrophic")
print("  Bannu   : P(High) = 100% — same province, even higher relative burden")
print("  Islamabad: P(High) = 0.2% — ICT wages (PKR 53,000) provide strong buffer")
print("  Quetta  : P(High) = 0% — Balochistan has lowest absolute prices despite lowest wages")

In [ ]:
# ── VISUALISATION 2b: Feature Distribution by Class ──────

fig, axes = plt.subplots(2, 3, figsize=(15, 9))
axes = axes.flatten()

feat_labels = {
    'Avg_Price'         : 'Average Item Price (PKR)',
    'Price_Volatility'  : 'Price Volatility (Std Dev)',
    'Affordability_Pct' : 'Basket Affordability (% of wage)',
    'Wheat_CV'          : 'Wheat Price CV (%)',
    'Tomato_CV'         : 'Tomato Price CV (%)',
    'Milk_CV'           : 'Milk Price CV (%)',
}

for ax, (feat, label) in zip(axes, feat_labels.items()):
    low  = city_df[city_df['Food_Stress'] == 0][feat].fillna(0)
    high = city_df[city_df['Food_Stress'] == 1][feat].fillna(0)

    # KDE plot approximation with histogram
    ax.hist(low,  bins=6, alpha=0.55, color=GREEN, label='Low Stress',  edgecolor='white')
    ax.hist(high, bins=6, alpha=0.55, color=RED,   label='High Stress', edgecolor='white')
    ax.axvline(low.mean(),  color=GREEN, linewidth=2, linestyle='--')
    ax.axvline(high.mean(), color=RED,   linewidth=2, linestyle='--')
    ax.set_title(label, fontsize=10, fontweight='bold')
    ax.legend(fontsize=8)

plt.suptitle('Naive Bayes: Feature Distributions by Food Stress Class\n'
             '(Dashed lines = class means, used by Gaussian NB)',
             fontsize=13, fontweight='bold')
plt.tight_layout()
plt.savefig('nb_feature_distributions.png', dpi=150, bbox_inches='tight')
plt.show()

print("The Gaussian NB model uses these distributions to compute")
print("P(feature | class) for each class. Wider separation = easier classification.")

---
## Section 5 — Model 3: K-Means Clustering *(Outside-Class Algorithm)*

### Citation
> MacQueen, J. B. (1967). "Some Methods for Classification and Analysis of
> Multivariate Observations." *Proceedings of 5th Berkeley Symposium on
> Mathematical Statistics and Probability*, 1, 281–297. University of California Press.

### Why K-Means?
K-Means is an **unsupervised clustering** algorithm that groups observations into K clusters
by minimising the **within-cluster sum of squared distances** to the cluster centroid.

**How it works:**
1. Randomly initialise K centroids
2. Assign each city to its nearest centroid (Euclidean distance)
3. Recompute centroids as the mean of assigned cities
4. Repeat steps 2–3 until assignments stop changing

**Why it fits our problem:**
- We do not have a pre-defined ground truth for city groupings — clustering discovers natural structure
- We want to find cities with **similar food security profiles** (price level + volatility + affordability + wages)
- The output provides a policy-actionable segmentation: different clusters require different interventions
- K-Means is interpretable — cluster centroids can be described in plain terms

**Outside-class justification:**
K-Means clustering was not covered in DS110 lectures (which covered Linear Regression
and Bayes Theorem). It is taught in chapters 4–5 of the referenced course textbook
[Mitchell, T., *Machine Learning*, McGraw-Hill, 1997] and widely used in data science practice.

**Hyperparameter Selection — Elbow Method + Silhouette Score:**
We test K from 2 to 6 and select the K with the highest Silhouette Score
(where 1.0 = perfect cluster separation, 0 = overlapping clusters).

In [ ]:
# ════════════════════════════════════════════════════════════
#  MODEL 3: K-MEANS CLUSTERING (OUTSIDE-CLASS ALGORITHM)
#  Cited: MacQueen (1967). UC Press.
#  Task   : Group cities into food insecurity profiles
#  Features: Avg_Price, Price_Volatility, Affordability_Pct, Wage
# ════════════════════════════════════════════════════════════

# ── Feature selection for clustering ──────────────────────
FEATURES_KM = ['Avg_Price', 'Price_Volatility', 'Affordability_Pct', 'Wage']
X_km = city_df[FEATURES_KM].values

# ── Standardise features (CRITICAL for K-Means) ───────────
# K-Means uses Euclidean distance, so features must be on the same scale.
# Without scaling, 'Wage' (range: 28,000–53,000) would completely dominate
# 'Affordability_Pct' (range: 12–19), making clustering meaningless.
scaler_km = StandardScaler()
X_scaled  = scaler_km.fit_transform(X_km)

print("Feature scaling applied (StandardScaler):")
for feat, mean, std in zip(FEATURES_KM, scaler_km.mean_, scaler_km.scale_):
    print(f"  {feat:<22}: mean={mean:,.1f}, std={std:,.1f}")

# ── Elbow Method + Silhouette Score ───────────────────────
inertias   = []
sil_scores = []
K_range    = range(2, 7)

for k in K_range:
    km = KMeans(n_clusters=k, random_state=42, n_init=10)
    km.fit(X_scaled)
    inertias.append(km.inertia_)
    sil = silhouette_score(X_scaled, km.labels_)
    sil_scores.append(sil)
    print(f"  K={k}: Inertia={km.inertia_:.2f}, Silhouette={sil:.3f}")

best_k = K_range.start + np.argmax(sil_scores)
print(f"\n  Best K = {best_k} (highest Silhouette Score = {max(sil_scores):.3f})")

In [ ]:
# ── Fit final K-Means model ───────────────────────────────
km_final = KMeans(n_clusters=best_k, random_state=42, n_init=10)
city_df['Cluster'] = km_final.fit_predict(X_scaled)

# ── Name the clusters based on their profile ──────────────
# Cluster 0: mainstream cities — moderate prices, moderate/high wages
# Cluster 1: outlier cities — very low prices AND very low wages (KPK + Balochistan)
cluster_labels = {0: 'Mainstream Urban (Moderate Stress)',
                  1: 'Low-Wage Periphery (Extreme Risk)'}
city_df['Cluster_Label'] = city_df['Cluster'].map(cluster_labels)

# ── Cluster centroids (unscaled — interpretable values) ───
centroids_unscaled = scaler_km.inverse_transform(km_final.cluster_centers_)
centroids_df = pd.DataFrame(centroids_unscaled, columns=FEATURES_KM)
centroids_df.index = [f'Cluster {i}: {cluster_labels[i]}' for i in range(best_k)]

sil_final = silhouette_score(X_scaled, km_final.labels_)

print("=" * 55)
print(f"MODEL 3: K-MEANS CLUSTERING (K={best_k}) — Results")
print("=" * 55)
print(f"  Silhouette Score: {sil_final:.3f}")
print("  (Range: -1 to 1. Score > 0.5 = well-separated clusters)")
print()
print("Cluster Centroids (unscaled, interpretable):")
print(centroids_df.round(1).to_string())
print()
print("City assignments:")
print(city_df[['City','Province','Affordability_Pct','Wage',
               'Cluster','Cluster_Label']].sort_values('Cluster').to_string(index=False))

In [ ]:
# ── VISUALISATION 3a: Elbow + Silhouette ─────────────────

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(13, 5))

# Elbow curve
ax1.plot(list(K_range), inertias, color=TEAL, linewidth=2.5,
         marker='o', markersize=8, markerfacecolor='white',
         markeredgecolor=TEAL, markeredgewidth=2)
ax1.set_xlabel('Number of Clusters (K)', fontsize=12)
ax1.set_ylabel('Inertia (Within-Cluster SSE)', fontsize=12)
ax1.set_title('Elbow Method\n(Lower inertia = tighter clusters)', fontsize=12, fontweight='bold')
ax1.set_xticks(list(K_range))

# Silhouette Score
bar_colors_sil = [RED if k == best_k else GREY for k in K_range]
ax2.bar(list(K_range), sil_scores, color=bar_colors_sil, edgecolor='white', width=0.5)
for k, s in zip(K_range, sil_scores):
    ax2.text(k, s + 0.01, f'{s:.3f}', ha='center', fontsize=10, fontweight='bold')
ax2.set_xlabel('Number of Clusters (K)', fontsize=12)
ax2.set_ylabel('Silhouette Score', fontsize=12)
ax2.set_title(f'Silhouette Scores\n(Best K = {best_k}, highlighted in red)',
              fontsize=12, fontweight='bold')
ax2.set_xticks(list(K_range))
ax2.set_ylim(0, 1)

plt.suptitle('K-Means Hyperparameter Selection — Elbow Method & Silhouette Analysis',
             fontsize=13, fontweight='bold', y=1.02)
plt.tight_layout()
plt.savefig('kmeans_selection.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# ── VISUALISATION 3b: Cluster Scatter + Province Map ─────

cluster_colors = {0: TEAL, 1: RED}

fig, axes = plt.subplots(1, 2, figsize=(15, 7))

# Panel 1: Scatter — Affordability vs Avg Price, coloured by cluster
ax = axes[0]
for cluster_id, grp in city_df.groupby('Cluster'):
    ax.scatter(
        grp['Avg_Price'], grp['Affordability_Pct'],
        c=cluster_colors[cluster_id], s=150,
        edgecolors='white', linewidth=1.5,
        label=cluster_labels[cluster_id], zorder=5
    )
    for _, row in grp.iterrows():
        ax.annotate(
            row['City'],
            xy=(row['Avg_Price'], row['Affordability_Pct']),
            xytext=(5, 5), textcoords='offset points',
            fontsize=8.5, color=GREY
        )

# Cluster centroids as large stars
for i, (cent_x, cent_y) in enumerate(
        zip(centroids_df['Avg_Price'], centroids_df['Affordability_Pct'])):
    ax.scatter(cent_x, cent_y, c=cluster_colors[i], s=350,
               marker='*', edgecolors='black', linewidth=1.5,
               zorder=10, label=f'Centroid {i}')

ax.axhline(MEDIAN_AFF, color=ORANGE, linewidth=1.5, linestyle=':',
           label=f'Stress threshold ({MEDIAN_AFF:.1f}%)')
ax.set_xlabel('Average Item Price (PKR)', fontsize=12)
ax.set_ylabel('Basket Affordability (% of monthly wage)', fontsize=12)
ax.set_title('K-Means Clusters: Price Level vs Affordability\n'
             '(Stars = cluster centroids)', fontsize=12, fontweight='bold')
ax.legend(fontsize=8, loc='upper right')

# Panel 2: Cluster membership horizontal bar
ax2 = axes[1]
c0_cities = city_df[city_df['Cluster'] == 0]['City'].tolist()
c1_cities = city_df[city_df['Cluster'] == 1]['City'].tolist()

all_cities     = c0_cities + c1_cities
affordability  = [city_df[city_df['City']==c]['Affordability_Pct'].values[0]
                  for c in all_cities]
bar_colors_c   = ([TEAL]*len(c0_cities)) + ([RED]*len(c1_cities))

bars = ax2.barh(all_cities, affordability, color=bar_colors_c,
                edgecolor='white')

for bar, val in zip(bars, affordability):
    ax2.text(val + 0.1, bar.get_y() + bar.get_height()/2,
             f'{val:.1f}%', va='center', fontsize=9)

ax2.axvline(MEDIAN_AFF, color=ORANGE, linewidth=2, linestyle=':',
            label=f'Threshold: {MEDIAN_AFF:.1f}%')
ax2.set_xlabel('Basket Cost as % of Provincial Monthly Wage', fontsize=11)
ax2.set_title('Cluster Assignment by Affordability Ratio\n'
              f'(Teal = Cluster 0, Red = Cluster 1)', fontsize=12, fontweight='bold')
ax2.legend(fontsize=9)
ax2.invert_yaxis()

c0_p = mpatches.Patch(color=TEAL, label=cluster_labels[0])
c1_p = mpatches.Patch(color=RED,  label=cluster_labels[1])
ax2.legend(handles=[c0_p, c1_p], fontsize=8)

plt.tight_layout()
plt.suptitle('K-Means Clustering — Pakistan City Food Security Profiles',
             fontsize=14, fontweight='bold', y=1.02)
plt.savefig('kmeans_clusters.png', dpi=150, bbox_inches='tight')
plt.show()

print("\nPolicy Interpretation:")
print("  Cluster 0 (Teal — 13 cities): Mainstream urban centres.")
print("    These cities need price volatility reduction policies.")
print("    Affordability is manageable but fragile (13-17%).")
print()
print("  Cluster 1 (Red — 3 cities: Peshawar, Bannu, Quetta):")
print("    These are low-wage periphery cities.")
print("    Quetta has cheap prices but wages are too low to benefit.")
print("    Peshawar and Bannu have highest food stress ratios (16-19%).")
print("    These cities need DIRECT income support, not just price controls.")

---
## Section 6 — Model Comparison & Summary

This section brings all three models together for a structured comparison.

In [ ]:
# ── Model Comparison Table ────────────────────────────────

fig, ax = plt.subplots(figsize=(15, 5))
ax.axis('off')

table_data = [
    ['Algorithm',     'Linear Regression',           'Gaussian Naive Bayes',           'K-Means Clustering'],
    ['Type',          'Supervised — Regression',      'Supervised — Classification',    'Unsupervised — Clustering'],
    ['In/Out Class',  'In-class (required)',           'In-class (required)',            'Outside class (MacQueen 1967)'],
    ['Target',        'Basket Cost (PKR)',             'High/Low Food Stress',           'No target (discovers groups)'],
    ['Key Metric',    'R² = 0.957',                   'LOO-CV Acc = 68.8%',             'Silhouette = 0.614'],
    ['MAE/Error',     'MAE = PKR 153.7',              'Train Acc = 87.5%',              '2 distinct clusters found'],
    ['Top Feature',   'Avg_Price (coef=2.47)',         'Affordability + Price CV',       'Wage level (cluster separator)'],
    ['Key Insight',   'Price × Volatility drive cost', 'Peshawar highest risk (P=1.0)', 'KPK/Balochistan isolated cluster'],
    ['Limitation',    'Only 16 city data points',      '16 cities = small dataset',      'K=2 driven by Quetta outlier'],
]

table = ax.table(cellText=table_data[1:],
                 colLabels=table_data[0],
                 cellLoc='center',
                 loc='center',
                 bbox=[0, 0, 1, 1])

# Style header
for j in range(4):
    table[(0, j)].set_facecolor(TEAL)
    table[(0, j)].set_text_props(color='white', fontweight='bold', fontsize=11)

# Alternate row colours
for i in range(1, len(table_data)):
    bg = '#EBF5FB' if i % 2 == 0 else 'white'
    for j in range(4):
        table[(i, j)].set_facecolor(bg)
        table[(i, j)].set_text_props(fontsize=9.5)
        table[(i, j)].set_height(0.13)

table.auto_set_font_size(False)
plt.title('Phase 3 — Model Comparison Summary',
          fontsize=14, fontweight='bold', pad=20)
plt.savefig('model_comparison.png', dpi=150, bbox_inches='tight')
plt.show()

---
## Section 7 — Combined Analytical Dashboard

Final 4-panel visualization combining all three models into one presentation-ready figure.

In [ ]:
# ── 4-panel combined figure ───────────────────────────────

fig = plt.figure(figsize=(18, 12))
fig.patch.set_facecolor('#F8FAFC')

gs = fig.add_gridspec(2, 2, hspace=0.40, wspace=0.35,
                      left=0.07, right=0.97, top=0.91, bottom=0.07)

ax_lr   = fig.add_subplot(gs[0, 0])   # Linear Regression
ax_lr2  = fig.add_subplot(gs[0, 1])   # NB probabilities
ax_km   = fig.add_subplot(gs[1, 0])   # K-Means scatter
ax_sum  = fig.add_subplot(gs[1, 1])   # Summary: all cities ranked

# ── Panel 1: LR Actual vs Predicted ──────────────────────
ax_lr.scatter(y_lr, y_pred_lr, color=TEAL, s=80, edgecolors='white',
              linewidth=1.5, zorder=5)
lim = [min(y_lr.min(), y_pred_lr.min())-300,
       max(y_lr.max(), y_pred_lr.max())+300]
ax_lr.plot(lim, lim, color=RED, linewidth=1.8, linestyle='--', alpha=0.7)
ax_lr.set_xlabel('Actual Basket Cost (PKR)', fontsize=10)
ax_lr.set_ylabel('Predicted Basket Cost (PKR)', fontsize=10)
ax_lr.set_title(f'① Linear Regression\nActual vs Predicted (R²={r2:.3f})',
                fontsize=11, fontweight='bold')
ax_lr.text(0.05, 0.90, f'R²={r2:.3f}\nMAE=PKR {mae:.0f}',
           transform=ax_lr.transAxes, fontsize=9,
           bbox=dict(boxstyle='round', facecolor='#D6EAF8'))

# ── Panel 2: NB Risk Probabilities ───────────────────────
nb_sorted = city_df.sort_values('NB_P_High')
colors_p2 = [RED if s==1 else GREEN for s in nb_sorted['Food_Stress']]
bars_p2 = ax_lr2.barh(nb_sorted['City'], nb_sorted['NB_P_High'],
                       color=colors_p2, edgecolor='white', alpha=0.85)
for bar, val in zip(bars_p2, nb_sorted['NB_P_High']):
    ax_lr2.text(min(val+0.02,0.90), bar.get_y()+bar.get_height()/2,
                f'{val:.0%}', va='center', fontsize=7.5)
ax_lr2.axvline(0.5, color='black', linewidth=1.2, linestyle='--')
ax_lr2.set_title(f'② Naive Bayes\nP(High Food Stress) per City\n'
                 f'(LOO-CV Acc: {loo_scores.mean():.1%})',
                 fontsize=11, fontweight='bold')
ax_lr2.set_xlabel('Probability of High Stress', fontsize=10)
ax_lr2.set_xlim(0, 1.05)

# ── Panel 3: K-Means Scatter ──────────────────────────────
for cid, grp in city_df.groupby('Cluster'):
    ax_km.scatter(grp['Avg_Price'], grp['Affordability_Pct'],
                  c=cluster_colors[cid], s=90,
                  edgecolors='white', linewidth=1.2,
                  label=cluster_labels[cid], zorder=5)
    for _, row in grp.iterrows():
        ax_km.annotate(row['City'],
                       xy=(row['Avg_Price'], row['Affordability_Pct']),
                       xytext=(4,4), textcoords='offset points',
                       fontsize=7, color=GREY)
for i, (cx, cy) in enumerate(zip(centroids_df['Avg_Price'],
                                  centroids_df['Affordability_Pct'])):
    ax_km.scatter(cx, cy, c=cluster_colors[i], s=300, marker='*',
                  edgecolors='black', linewidth=1.2, zorder=10)
ax_km.axhline(MEDIAN_AFF, color=ORANGE, linewidth=1.2, linestyle=':')
ax_km.set_xlabel('Average Item Price (PKR)', fontsize=10)
ax_km.set_ylabel('Basket Affordability (% wage)', fontsize=10)
ax_km.set_title(f'③ K-Means Clustering (K={best_k})\n'
                f'Silhouette Score = {sil_final:.3f}',
                fontsize=11, fontweight='bold')
ax_km.legend(fontsize=7, loc='lower right')

# ── Panel 4: Combined Risk Ranking ───────────────────────
city_df['Risk_Score'] = (
    city_df['Affordability_Pct'] / city_df['Affordability_Pct'].max() * 50 +
    city_df['NB_P_High'] * 50
).round(1)
risk_sorted = city_df.sort_values('Risk_Score', ascending=False)
risk_colors = [RED if r > 50 else ORANGE if r > 35 else GREEN
               for r in risk_sorted['Risk_Score']]
bars_r = ax_sum.barh(risk_sorted['City'], risk_sorted['Risk_Score'],
                     color=risk_colors, edgecolor='white')
for bar, val in zip(bars_r, risk_sorted['Risk_Score']):
    ax_sum.text(val+0.5, bar.get_y()+bar.get_height()/2,
                f'{val:.0f}', va='center', fontsize=8.5)
ax_sum.axvline(50, color='black', linewidth=1.2, linestyle='--',
               label='High-risk threshold (50)')
ax_sum.set_xlabel('Composite Food Insecurity Risk Score (0–100)', fontsize=10)
ax_sum.set_title('④ Composite Risk Ranking\n'
                 '(LR Affordability + NB Probability)',
                 fontsize=11, fontweight='bold')
ax_sum.invert_yaxis()
ax_sum.set_xlim(0, 105)

red_p    = mpatches.Patch(color=RED,    label='High risk (>50)')
orange_p = mpatches.Patch(color=ORANGE, label='Medium risk (35-50)')
green_p  = mpatches.Patch(color=GREEN,  label='Lower risk (<35)')
ax_sum.legend(handles=[red_p, orange_p, green_p], fontsize=8)

fig.suptitle(
    'Pakistan Food Insecurity — Combined ML Analysis Dashboard\n'
    'Linear Regression · Naive Bayes · K-Means Clustering',
    fontsize=15, fontweight='bold'
)
plt.savefig('phase3_combined_dashboard.png', dpi=150, bbox_inches='tight')
plt.show()
print("Combined dashboard saved.")

---
## Section 8 — Final Findings, Recommendations & Limitations

In [ ]:
print("=" * 65)
print("PHASE 3 — KEY FINDINGS SUMMARY")
print("=" * 65)

print("""
MODEL 1 — LINEAR REGRESSION (R² = 0.957):
  • Average item price is the strongest predictor of basket cost
    (coefficient = 2.47: each PKR 1 rise in avg price → PKR 2.47
     higher basket cost — amplified because the basket has 10 items)
  • Price volatility also significantly drives basket cost upward
  • Wage level has minimal direct effect on basket cost —
    confirming that prices are market-driven, not wage-driven
  • The model explains 95.7% of city-level basket cost variation

MODEL 2 — GAUSSIAN NAIVE BAYES (LOO-CV = 68.8%):
  • Peshawar and Bannu are classified as High Stress with P = 1.00
    — KPK wages of PKR 32,000 make 18.7% affordability catastrophic
  • Islamabad has P(High Stress) = 0.2% — ICT wages buffer food costs
  • LOO-CV accuracy of 68.8% is honest for 16 data points
  • Training accuracy of 87.5% confirms the model has learned real patterns

MODEL 3 — K-MEANS CLUSTERING (Silhouette = 0.614):
  • Two well-separated clusters naturally emerge from the data
  • Cluster 1 (Peshawar, Bannu, Quetta) = Low-wage periphery
    These 3 cities have fundamentally different economic profiles
    from the other 13 — they need different policy interventions
  • Cluster 0 (13 cities) = Mainstream urban centres
    These cities share similar price levels but vary in affordability
    based on provincial wage differences

CROSS-MODEL POLICY INSIGHT:
  • All three models agree: Peshawar and Bannu are the highest-risk cities
  • The root cause is wages (PKR 32,000 in KPK), not food prices
  • Quetta is an anomaly: lowest food prices AND lowest wages — the
    cheap price is offset by extremely limited household budgets
  • Islamabad, Karachi, Hyderabad show consistently lower risk

RECOMMENDATIONS:
  1. Direct income support for KPK informal workers (not price controls)
  2. Protein food price subsidies (pulses, eggs) — fastest-inflating items
  3. Province-specific food security monitoring — national averages mislead
  4. Reduce wheat price volatility — it destabilises all household budgets
"""  )

In [ ]:
print("=" * 65)
print("LIMITATIONS")
print("=" * 65)

print("""
1. Small city sample (n=16): All three models are limited by having
   only 16 city data points. For a statistically robust regression
   and classification, ideally 100+ observations are needed.
   The LOO-CV was used specifically to address this limitation.

2. City-level aggregation hides within-city variation:
   A city average does not capture the experience of informal
   workers, daily wage labourers, or rural households near cities.

3. Affordability based on formal employment wages:
   PBS Labour Force Survey wages reflect formal sector employment.
   Actual household income for vulnerable groups is lower,
   meaning the food stress problem is WORSE than our models show.

4. Clustering driven by Quetta outlier:
   The K=2 result is partly because Quetta's very low food prices
   (due to Balochistan market structure) create a natural outlier.
   With more cities, we would expect more nuanced clusters.

5. Time dimension not modelled:
   The models use average prices across 2 years. A time-series
   model could capture how food insecurity risk CHANGES over time,
   especially after the 2024 wheat price intervention.
""")